# Hardware Expressivity — IBM Quantum (Oriol)

This notebook runs ORS and QELM experiments on IBM Aachen quantum hardware via Qiskit Runtime. It is a companion to the main hardware notebook (`Exp - Quantum Hardware.ipynb`) focused on circuit construction and submission.

## Infrastructure Setup

The first code block defines hardware interface utilities:
- **`qubit_error_scores()`** — reads single-qubit readout errors and two-qubit gate errors from backend calibration data.
- **`coupling_adjacency()`** — builds the qubit connectivity graph from the backend coupling map.
- **`grow_block(seed, ...)`** — greedy BFS starting from the lowest-error qubit, adding neighbours with minimum gate error at each step.
- **`find_disjoint_blocks()`** — finds multiple low-error connected $n$-qubit subgraphs separated by a buffer of unused qubits, enabling parallel independent experiments on one chip.
- **`split_counts_per_block()`** — parses Qiskit's multi-register bitstring counts back into per-block measurement outcomes.

In [1]:
import math
import numpy as np
from collections import defaultdict

from qiskit import QuantumCircuit, ClassicalRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeAachen
from metrics import build_parallel_circuit

# ============================================================
# 2. Qubit layout: find disjoint n-qubit connected subgraphs
# ============================================================
def qubit_error_scores(backend):
    """
    Return:
      readout_err[q] : float
      edge_err[(a,b)] : float  (symmetric, lower = better)
    """
    props = backend.properties()
    readout_err = {}
    for q in range(backend.num_qubits):
        try:
            readout_err[q] = props.readout_error(q)
        except Exception:
            readout_err[q] = 0.05  # fallback

    # 2Q gate errors: look for ecr / cz / cx (Heron uses cz; older devices ecr/cx)
    edge_err = {}
    for gate in props.gates:
        if gate.gate in ("cz", "ecr", "cx") and len(gate.qubits) == 2:
            err = None
            for p in gate.parameters:
                if p.name == "gate_error":
                    err = p.value
                    break
            if err is None:
                continue
            a, b = gate.qubits
            edge_err[(a, b)] = err
            edge_err[(b, a)] = err
    return readout_err, edge_err


def coupling_adjacency(backend):
    """Undirected adjacency dict from the coupling map."""
    adj = defaultdict(set)
    for a, b in backend.coupling_map.get_edges():
        adj[a].add(b)
        adj[b].add(a)
    return adj


def grow_block(seed, adj, used, size, edge_err):
    """
    Greedy BFS from `seed`: at each step add the neighbor (of the current block)
    with the lowest gate error to any block member. Returns block of `size`
    qubits or None if not possible.
    """
    if seed in used:
        return None
    block = {seed}
    frontier = set(n for n in adj[seed] if n not in used)

    while len(block) < size:
        if not frontier:
            return None
        # pick frontier qubit with min edge_err to any block member
        best_q, best_err = None, math.inf
        for q in frontier:
            local = min(
                (edge_err.get((q, b), math.inf) for b in block if b in adj[q]),
                default=math.inf,
            )
            if local < best_err:
                best_err, best_q = local, q
        if best_q is None or best_err == math.inf:
            return None
        block.add(best_q)
        frontier.discard(best_q)
        for nb in adj[best_q]:
            if nb not in used and nb not in block:
                frontier.add(nb)
    return block


def score_block(block, adj, edge_err, readout_err):
    """Sum of intra-block edge errors + readout errors. Lower is better."""
    s = sum(readout_err[q] for q in block)
    seen = set()
    for q in block:
        for nb in adj[q]:
            if nb in block and (nb, q) not in seen:
                s += edge_err.get((q, nb), 1.0)
                seen.add((q, nb))
    return s


def find_disjoint_blocks(backend, n_qubits=16, max_blocks=10, buffer=1):
    """
    Greedily find disjoint connected n_qubits-size blocks separated by `buffer`
    unused qubits (buffer=1 means the immediate neighbors of any chosen qubit
    are forbidden from other blocks).
    """
    readout_err, edge_err = qubit_error_scores(backend)
    adj = coupling_adjacency(backend)
    all_qubits = list(range(backend.num_qubits))
    # seed order: lowest-readout-error qubits first
    seeds = sorted(all_qubits, key=lambda q: readout_err[q])

    used = set()
    blocks = []

    for seed in seeds:
        if len(blocks) >= max_blocks:
            break
        if seed in used:
            continue
        block = grow_block(seed, adj, used, n_qubits, edge_err)
        if block is None:
            continue
        # mark block + buffer ring as used
        used |= block
        if buffer > 0:
            ring = set()
            frontier = set(block)
            for _ in range(buffer):
                new_frontier = set()
                for q in frontier:
                    for nb in adj[q]:
                        if nb not in block and nb not in ring:
                            ring.add(nb)
                            new_frontier.add(nb)
                frontier = new_frontier
            used |= ring
        blocks.append(sorted(block))

    # Print summary
    for i, blk in enumerate(blocks):
        s = score_block(blk, adj, edge_err, readout_err)
        print(f"  Block {i}: {blk}  (score={s:.4f})")
    return blocks

def split_counts_per_block(counts, blocks):
    """
    Qiskit count keys are written with whitespace separating classical registers,
    in REGISTER-REVERSED order: 'c{N-1} c{N-2} ... c0'. Each register's bits
    are also little-endian within itself. We just split on whitespace and use
    the natural ordering of each substring.
    Returns list of dicts {bitstring -> count} per block.
    """
    per_block = [defaultdict(int) for _ in blocks]
    for key, n in counts.items():
        parts = key.split()  # length = len(blocks), reversed register order
        # parts[0] is the LAST added register, parts[-1] is the first
        parts = list(reversed(parts))
        for i, bits in enumerate(parts):
            per_block[i][bits] += n
    return per_block

## Experiment 1: Noisy ORS on IBM Hardware

We measure the Order Statistics Score ($\Lambda$) on circuits executed directly on IBM Aachen. Unlike noiseless simulation, hardware ORS is affected by:
- Gate errors (depolarising, over-rotation, crosstalk)
- Readout errors

The ORS gap is compared against the **noisy Haar baseline** $\Lambda^{\mathrm{Haar},\mathrm{noisy}}(f)$, where $f$ is estimated from the per-block gate-error budget. Families tested: G1 (Clifford), G3 (non-Clifford), D2_random across block sizes $n \in \{6, 10, 12\}$.

### n=12 Qubit Blocks

We select 12-qubit connected subgraphs on IBM Aachen using `find_disjoint_blocks(buffer=2)`. Circuits are built in parallel across all available blocks to maximise hardware utilisation.

**Gate-count sweep**: 20, 50, 100, 200, 500 (converted to circuit depth via `depth = round(num_gates / 12)`).

In [24]:
import pickle
from metrics import build_parallel_circuit
from qiskit_ibm_runtime.fake_provider import FakeAachen


nqbits = 12
max_blocks = 10
buffer = 2
nshots = 30000

backend = FakeAachen()
blocks = find_disjoint_blocks(backend, n_qubits=nqbits, max_blocks=max_blocks, buffer=buffer)

# num_gates -> depth (num_gates = nqbits * depth)
specs = []
for ng in [20, 50, 100, 200, 500]:
    d = max(1, round(ng / 12))
    specs.append(("G1", {"depth": d}, f"G1_ng{ng}"))
    specs.append(("G3", {"depth": d}, f"G3_ng{ng}"))
for deg in [1, 2, 4]:
    specs.append(("D2_random", {"expected_degree": deg}, f"D2random_deg{deg}"))

circuits = []
for family, kwargs, label in specs:
    qc, _ = build_parallel_circuit(backend, blocks, family, seed=0, **kwargs)
    circuits.append({"label": label, "family": family, "kwargs": kwargs, "qc": qc})
    print(f"{label:18s}  depth={qc.depth()}")

with open("circuits_12qubits_ORS.pkl", "wb") as f:
    pickle.dump(circuits, f)

print(f"\nStored {len(circuits)} circuits in 'circuits' list and 'circuits_12qubits_ORS.pkl'")

  Block 0: [3, 4, 5, 7, 16, 17, 23, 24, 25, 26, 27, 28]  (score=0.0670)
  Block 1: [11, 12, 13, 14, 15, 18, 31, 32, 33, 34, 39, 53]  (score=0.0896)
  Block 2: [80, 81, 82, 83, 96, 100, 101, 102, 103, 116, 120, 121]  (score=0.1453)
  Block 3: [36, 40, 41, 42, 43, 44, 56, 63, 64, 65, 66, 77]  (score=0.1092)
  Block 4: [90, 91, 98, 110, 111, 112, 113, 114, 115, 119, 132, 133]  (score=0.1127)
  Block 5: [127, 128, 136, 137, 140, 141, 142, 143, 144, 145, 146, 147]  (score=0.3262)
G1_ng20             depth=9
G3_ng20             depth=9
G1_ng50             depth=16
G3_ng50             depth=16
G1_ng100            depth=25
G3_ng100            depth=25
G1_ng200            depth=52
G3_ng200            depth=52
G1_ng500            depth=112
G3_ng500            depth=112
D2random_deg1       depth=5
D2random_deg2       depth=5
D2random_deg4       depth=8

Stored 13 circuits in 'circuits' list and 'circuits_12qubits_ORS.pkl'


### n=10 Qubit Blocks

Same protocol as $n=12$ but with 10-qubit blocks. More blocks fit on the chip, providing more independent ORS samples per hardware run and lower statistical noise in the aggregate $\Lambda$ estimate.

**Gate-count sweep**: 5, 20, 50, 100, 200, 300.

In [25]:
import pickle
from metrics import build_parallel_circuit
from qiskit_ibm_runtime.fake_provider import FakeAachen


nqbits = 10
max_blocks = 10
buffer = 2
nshots = 30000

backend = FakeAachen()
blocks = find_disjoint_blocks(backend, n_qubits=nqbits, max_blocks=max_blocks, buffer=buffer)

# num_gates -> depth (num_gates = nqbits * depth)
specs = []
for ng in [5, 20, 50, 100, 200, 300]:
    d = max(1, round(ng / 12))
    specs.append(("G1", {"depth": d}, f"G1_ng{ng}"))
    specs.append(("G3", {"depth": d}, f"G3_ng{ng}"))
for deg in [1, 2, 4]:
    specs.append(("D2_random", {"expected_degree": deg}, f"D2random_deg{deg}"))

circuits = []
for family, kwargs, label in specs:
    qc, _ = build_parallel_circuit(backend, blocks, family, seed=0, **kwargs)
    circuits.append({"label": label, "family": family, "kwargs": kwargs, "qc": qc})
    print(f"{label:18s}  depth={qc.depth()}")

with open("circuits_10qubits_ORS.pkl", "wb") as f:
    pickle.dump(circuits, f)

print(f"\nStored {len(circuits)} circuits in 'circuits' list and 'circuits_10qubits_ORS.pkl'")

  Block 0: [3, 7, 16, 17, 23, 24, 25, 26, 27, 28]  (score=0.0556)
  Block 1: [11, 12, 13, 14, 15, 18, 31, 32, 33, 34]  (score=0.0698)
  Block 2: [82, 83, 96, 100, 101, 102, 103, 116, 120, 121]  (score=0.1296)
  Block 3: [36, 40, 41, 42, 43, 44, 56, 63, 64, 65]  (score=0.0901)
  Block 4: [90, 91, 98, 110, 111, 112, 113, 114, 115, 119]  (score=0.0911)
  Block 5: [136, 137, 140, 141, 142, 143, 144, 145, 146, 147]  (score=0.3102)
  Block 6: [47, 48, 49, 50, 51, 52, 57, 58, 70, 71]  (score=0.1261)
  Block 7: [131, 135, 138, 139, 150, 151, 152, 153, 154, 155]  (score=0.1456)
G1_ng5              depth=6
G3_ng5              depth=6
G1_ng20             depth=8
G3_ng20             depth=8
G1_ng50             depth=18
G3_ng50             depth=18
G1_ng100            depth=26
G3_ng100            depth=26
G1_ng200            depth=47
G3_ng200            depth=47
G1_ng300            depth=72
G3_ng300            depth=72
D2random_deg1       depth=5
D2random_deg2       depth=6
D2random_deg4       dept

### n=6 Qubit Blocks

The smallest block size ($D = 64$). This connects directly to the $n=6$ simulation regime of Exp1, where KL and ORS were cross-validated — enabling a direct simulation-to-hardware comparison.

**Gate-count sweep**: 5, 20, 50, 100, 200, 300.

In [27]:
import pickle
from metrics import build_parallel_circuit
from qiskit_ibm_runtime.fake_provider import FakeAachen


nqbits = 6
max_blocks = 10
buffer = 2
nshots = 30000

backend = FakeAachen()
blocks = find_disjoint_blocks(backend, n_qubits=nqbits, max_blocks=max_blocks, buffer=buffer)

# num_gates -> depth (num_gates = nqbits * depth)
specs = []
for ng in [5, 20, 50, 100, 200, 300]:
    d = max(1, round(ng / 12))
    specs.append(("G1", {"depth": d}, f"G1_ng{ng}"))
    specs.append(("G3", {"depth": d}, f"G3_ng{ng}"))
for deg in [1, 2, 4]:
    specs.append(("D2_random", {"expected_degree": deg}, f"D2random_deg{deg}"))

circuits = []
for family, kwargs, label in specs:
    qc, _ = build_parallel_circuit(backend, blocks, family, seed=0, **kwargs)
    circuits.append({"label": label, "family": family, "kwargs": kwargs, "qc": qc})
    print(f"{label:18s}  depth={qc.depth()}")

with open("circuits_6qubits_ORS.pkl", "wb") as f:
    pickle.dump(circuits, f)

print(f"\nStored {len(circuits)} circuits in 'circuits' list and 'circuits_6qubits_ORS.pkl'")

  Block 0: [7, 17, 25, 26, 27, 28]  (score=0.0372)
  Block 1: [11, 18, 31, 32, 33, 34]  (score=0.0402)
  Block 2: [83, 96, 100, 101, 102, 103]  (score=0.0479)
  Block 3: [41, 42, 43, 44, 56, 63]  (score=0.0474)
  Block 4: [0, 1, 2, 3, 4, 16]  (score=0.1194)
  Block 5: [90, 91, 98, 110, 111, 112]  (score=0.0490)
  Block 6: [136, 140, 141, 142, 143, 144]  (score=0.0957)
  Block 7: [47, 48, 49, 50, 51, 52]  (score=0.0682)
  Block 8: [127, 128, 129, 130, 137, 147]  (score=0.1224)
  Block 9: [150, 151, 152, 153, 154, 155]  (score=0.0547)
G1_ng5              depth=5
G3_ng5              depth=5
G1_ng20             depth=8
G3_ng20             depth=8
G1_ng50             depth=12
G3_ng50             depth=12
G1_ng100            depth=25
G3_ng100            depth=25
G1_ng200            depth=46
G3_ng200            depth=46
G1_ng300            depth=64
G3_ng300            depth=64
D2random_deg1       depth=6
D2random_deg2       depth=7
D2random_deg4       depth=9

Stored 15 circuits in 'circuits'

### Submitting Circuits to Hardware

Circuits are submitted to IBM Aachen via Qiskit Runtime's `SamplerV2`. Each circuit is first transpiled to the native gate set ({rz, sx, x, cz}) at optimisation level 3.

After execution, raw measurement counts are parsed per block with `split_counts_per_block()`, then fed into `lambda_score_counts()` (ORS in count space, not probability space) for hardware-compatible expressivity evaluation.

In [3]:
# Run without error mitigation

## Experiment 2: QELM for LiH Ground-State Prediction

A **Quantum Extreme Learning Machine (QELM)** treats the circuit as a fixed feature map — no temporal feedback, no training of circuit parameters.

**Task**: predict the 5 excited-state energies of the LiH molecule from its ground-state wavefunction, sampled at 100 bond lengths.

**Protocol**:
1. Load 100 LiH ground states ($2^8 = 256$ amplitude coefficients)
2. Amplitude-encode each ground state onto 10 physical qubits via `angle_encode()`
3. Apply the reservoir circuit (G3 or G1, 500 gates)
4. Measure 2-local Pauli observables on each qubit block via `EstimatorV2`
5. Train a Ridge regressor on the observable expectations → excited-state energies

### Load LiH Dataset

The LiH dataset provides pre-computed quantum chemistry results (Jordan-Wigner encoding, active space) at 100 bond lengths subsampled from a dense grid.

- `ground_states`: shape (100, 256) — real-valued ground-state amplitudes
- `spectrums`: shape (100, 5) — excited-state energies in Hartree
- `bond_lengths`: shape (100,) — internuclear distance in Angstroms

In [2]:
with open('spectrums_LiH.npy', 'rb') as f:
            spectrums = np.load(f)
with open('bond_lengths_LiH.npy', 'rb') as f:
            bond_lengths = np.load(f)
with open('ground_states_LiH.npy', 'rb') as f:
            ground_states = np.load(f)

num_samples = 100
idxs = np.linspace(0, ground_states.shape[0] - 1, num_samples).astype('int')
spectrums, bond_lengths, ground_states = spectrums[idxs], bond_lengths[idxs], ground_states[idxs]
print("Spectrums shape:", spectrums.shape)
print("Bond lengths shape:", bond_lengths.shape)
print("Ground states shape:", ground_states.shape)

Spectrums shape: (100, 5)
Bond lengths shape: (100,)
Ground states shape: (100, 256)


### Encoding and Observable Helpers

- **`angle_encode(amplitudes, qubits, qc)`** — maps each real amplitude $a$ to an Ry rotation $\theta = 2\arcsin(a)$ on the corresponding qubit, so $R_y(\theta)|0\rangle$ has the same $|1\rangle$-component as $|a|$. Preserves amplitude magnitudes faithfully for real-valued states.
- **`k_local_observables(k, block, total_qubits)`** — generates all $\binom{|\mathrm{block}|}{k} \times 3^k$ weight-$k$ Pauli strings supported on the given qubit block. These form the feature vector passed to the Ridge readout.

In [16]:
import numpy as np
from qiskit import QuantumCircuit
from itertools import combinations, product
from qiskit.quantum_info import SparsePauliOp
import pickle

def angle_encode(amplitudes, qubits, qc):
    """
    Encode `amplitudes` (length s) into `s` single-qubit Ry rotations on the
    given physical qubits of `qc`. One amplitude -> one qubit.
    
    Maps amplitude a -> angle 2*arcsin(a) so that Ry(angle)|0> has |1>-component
    equal to a. This preserves the magnitude faithfully; with real amplitudes
    in [-1, 1] no information is lost.
    """
    assert len(amplitudes) == len(qubits)
    for amp, q in zip(amplitudes, qubits):
        # clip for numerical safety; arcsin domain is [-1, 1]
        a = float(np.clip(np.real(amp), -1.0, 1.0))
        theta = 2.0 * np.arcsin(a)
        qc.ry(theta, q)

def k_local_observables(k, block, total_qubits, paulis=('X', 'Y', 'Z')):
        """Generate all k-local Pauli observables."""
        obs = []
        for sites in combinations(block, k):  # sites are PHYSICAL qubit indices
            for pauli_tuple in product(paulis, repeat=k):
                # Qiskit Pauli strings are little-endian: index 0 is the RIGHTMOST char
                op = ['I'] * total_qubits
                for site, pauli in zip(sites, pauli_tuple):
                    op[site] = pauli
                # reverse so index 0 -> rightmost
                obs.append(SparsePauliOp(''.join(reversed(op))))
        return obs

### G3 Reservoir Circuits

G3 (gate set {CNOT, H, T}) is expected to be near the Haar limit at 500 gates, providing high expressivity ($\Lambda \approx \Lambda^{\mathrm{Haar}}$) and rich feature coverage (high $R_{\mathrm{eff}}$).

Circuits are built in batches of `num_blocks` (one ground state per block), transpiled to the native IBM gate set, and serialised to `circuits_LiH_G3_500gates.pkl` for submission.

In [19]:
# Run circuit for all values of ground states:
gate_set = 'G3'
num_gates = 500
nshots = 20000
threshold = 1e-6
nqbits = 10

max_blocks = 5
buffer = 3

backend = FakeAachen()
blocks = find_disjoint_blocks(backend, n_qubits=nqbits, max_blocks=max_blocks, buffer=buffer)
num_blocks = len(blocks)
print(f'Total :{num_blocks}')

qc_list = []
qc2, _ = build_parallel_circuit(backend, blocks, gate_set, seed=0, depth = int(num_gates/nqbits))

for j in range(num_samples//num_blocks):
    qc = QuantumCircuit(backend.num_qubits)
    for i in range(len(blocks)):
        block = blocks[i]  
        state = ground_states[j*num_blocks + i]
        # Truncate
        mask = np.abs(state) > threshold
        amplitudes  = state[mask]
        # Initialize
        angle_encode(amplitudes, block, qc)
    qc = qc.compose(qc2)
    qc_list.append(qc)
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
    isa = pm.run(qc)
    print(f"\nCircuit {j}  original: depth={qc.depth():4d}  ops={dict(qc.count_ops())}")
    print(f"Circuit {j} transpilled: depth={isa.depth():4d}  ops={dict(isa.count_ops())}")

with open("circuits_LiH_G3_500gates.pkl", "wb") as f:
    pickle.dump(qc_list, f)

  Block 0: [3, 7, 16, 17, 23, 24, 25, 26, 27, 28]  (score=0.0556)
  Block 1: [82, 83, 96, 100, 101, 102, 103, 116, 120, 121]  (score=0.1296)
  Block 2: [40, 41, 42, 43, 56, 63, 64, 65, 66, 67]  (score=0.0833)
  Block 3: [11, 12, 13, 14, 15, 18, 19, 33, 34, 35]  (score=1.4067)
  Block 4: [90, 91, 98, 110, 111, 112, 113, 114, 115, 119]  (score=0.0911)
Total :5

Circuit 0  original: depth= 132  ops={'h': 842, 't': 840, 'cx': 818, 'ry': 50, 'measure': 50}
Circuit 0 transpilled: depth= 140  ops={'sx': 934, 'rz': 892, 'cz': 469, 'x': 56, 'measure': 50}

Circuit 1  original: depth= 132  ops={'h': 842, 't': 840, 'cx': 818, 'ry': 50, 'measure': 50}
Circuit 1 transpilled: depth= 142  ops={'sx': 934, 'rz': 889, 'cz': 469, 'x': 57, 'measure': 50}

Circuit 2  original: depth= 132  ops={'h': 842, 't': 840, 'cx': 818, 'ry': 50, 'measure': 50}
Circuit 2 transpilled: depth= 142  ops={'sx': 934, 'rz': 888, 'cz': 469, 'x': 53, 'measure': 50}

Circuit 3  original: depth= 132  ops={'h': 842, 't': 840, 'cx'

### Observable Pool and Estimator Setup

The measurement pool consists of all 2-local Pauli strings on each 10-qubit block: $\binom{10}{2} \times 9 = 405$ observables per block. Qiskit's `EstimatorV2` returns expectation values directly from the hardware measurements.

The full expectation-value vector across all blocks forms the **feature row** for the Ridge regression readout.

In [24]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
# Observables
observables = []
for block in blocks:
    obs = k_local_observables(2, block, backend.num_qubits)
    for o in obs:
        observables.append(o)

isa_observables = [obs.apply_layout(isa.layout) for obs in observables]

# Run via Estimator
estimator = Estimator(mode=backend)
estimator.options.default_shots = nshots
#job = estimator.run([(isa, isa_observables)])
#result = job.result()
#expvals = result[0].data.evs   # array of shape (len(observables),)
#stds = result[0].data.stds     # standard errors



### G1 Reservoir Circuits (Clifford Comparison)

G1 uses only Clifford gates ({CNOT, H, S}). Clifford circuits produce highly structured, non-Haar output distributions (low $\Lambda$). By running the same QELM protocol with G1, we test whether low expressivity leads to worse prediction accuracy, holding gate count, encoding, and observables fixed.

In [ ]:
# Run circuit for all values of ground states:
gate_set = 'G1'

qc_list = []
qc2, _ = build_parallel_circuit(backend, blocks, gate_set, seed=0, depth = int(num_gates/nqbits))
for j in range(num_samples//num_blocks):
    qc = QuantumCircuit(backend.num_qubits)
    for i in range(len(blocks)):
        block = blocks[i]  
        state = ground_states[j*num_blocks + i]
        # Truncate
        mask = np.abs(state) > threshold
        amplitudes  = state[mask]
        # Initialize
        angle_encode(amplitudes, block, qc)
    qc = qc.compose(qc2)
    qc_list.append(qc)
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
    isa = pm.run(qc)
    print(f"\nCircuit {j}  original: depth={qc.depth():4d}  ops={dict(qc.count_ops())}")
    print(f"Circuit {j} transpilled: depth={isa.depth():4d}  ops={dict(isa.count_ops())}")

with open("circuits_LiH_G1_500gates.pkl", "wb") as f:
    pickle.dump(qc_list, f)

  Block 0: [3, 7, 16, 17, 23, 24, 25, 26, 27, 28]  (score=0.0556)
  Block 1: [82, 83, 96, 100, 101, 102, 103, 116, 120, 121]  (score=0.1296)
  Block 2: [40, 41, 42, 43, 56, 63, 64, 65, 66, 67]  (score=0.0833)
  Block 3: [11, 12, 13, 14, 15, 18, 19, 33, 34, 35]  (score=1.4067)
  Block 4: [90, 91, 98, 110, 111, 112, 113, 114, 115, 119]  (score=0.0911)
Total :5

Circuit 0  original: depth= 132  ops={'h': 842, 'x': 840, 'cx': 818, 'ry': 50, 'measure': 50}
Circuit 0 transpilled: depth= 119  ops={'sx': 638, 'rz': 571, 'cz': 418, 'x': 77, 'measure': 50}
